# Clase 06 · Práctica guiada: qué son los outliers
Dataset: **viviendas en Lima** (sintético, con outliers puestos a propósito).

El profesor dio una clase **teórica**: qué es un outlier (el círculo entre triángulos), sus **dos caras**
(hallazgo en lo descriptivo, sesgo en lo predictivo), por qué **no se pueden eliminar todos**,
qué puede **significar** (error, límite, punto de interés) y las dos formas de tratarlo
(**eliminar** o **llevar a límites**). Después respondió preguntas sobre series de tiempo,
`PowerTransformer` y la V de Cramér.

Aquí llevamos cada idea a código. Todo el código es 📌 Extra (la clase no tuvo código).
Los métodos de **detección** (IQR, z-score) llegan en la clase 07: aquí solo usamos límites conocidos.

In [ ]:
import sys
from pathlib import Path

# Localiza la raíz del repo, para que el notebook funcione desde cualquier carpeta
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "framework").exists())
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd

from framework.testing import ruta_dataset  # si el CSV no existe, lo genera

# matplotlib es opcional: si no está instalado, las celdas de gráficos se saltan solas
try:
    import matplotlib.pyplot as plt
    HAY_PLT = True
except ImportError:
    HAY_PLT = False
    print("matplotlib no está instalado: se omiten los gráficos (pip install matplotlib)")

df = pd.read_csv(ruta_dataset("viviendas_lima.csv"))
df.shape

## Paso 1 · El círculo entre triángulos
Un outlier *"se desvía significativamente del resto"*. Primer vistazo con `describe()`:
mira la distancia entre el percentil 75 % y el máximo.

In [ ]:
df[["area_m2", "precio_usd", "antiguedad_anios"]].describe().round(1)

Fíjate:
- `area_m2`: el 75 % de las viviendas tiene ≤ 126 m², pero el máximo es **2746 m²**.
- `antiguedad_anios`: el 75 % tiene ≤ 16 años, pero el máximo es **150**.
- `precio_usd`: el 75 % cuesta ≤ 195 000 USD, pero el máximo pasa de **2 millones**.

Tres "círculos". Veamos de cerca cada uno.

In [ ]:
# Las 12 áreas más grandes: ¿parecen reales?
df.nlargest(12, "area_m2")[["tipo_inmueble", "distrito", "area_m2", "habitaciones", "precio_usd"]]

Un **departamento de 689 m² con 138 700 USD** no tiene sentido: es un **error de medición/digitación**
(el área se multiplicó por 10). El profesor: *"errores de medición [...] no hemos validado el instrumento"*.

In [ ]:
# Antigüedad: ¿hay valores imposibles?
df.loc[df["antiguedad_anios"] > 100, ["id", "tipo_inmueble", "antiguedad_anios"]]

In [ ]:
# Precio: las viviendas más caras
df.nlargest(5, "precio_usd")[["tipo_inmueble", "distrito", "area_m2", "precio_usd"]]

## Paso 2 · ¿Qué significa cada outlier?
| Caso | Significado | ¿Por qué? |
|---|---|---|
| Antigüedad 150 | **Error** | Imposible: como la "persona de 0 años" del profesor |
| Área ×10 | **Error** (o **límite**) | Un departamento de 689 m² a ese precio no existe |
| Casa de 2.1 millones | **Punto de interés** | Rara, pero posible: es el "Carlos" de las viviendas |

🧠 La decisión **no la toma pandas, la tomas tú** con conocimiento del negocio.

## Paso 3 · Llevar a límites (capping)
Cuando la variable tiene **límites conocidos**, el profesor los lleva a esos límites: edad < 0 → 0, edad > 125 → 125.
Para la antigüedad de una vivienda usamos 0 y 100 (decisión de negocio: las reales llegan a 60).

Primero contamos cuántos valores escapan; luego aplicamos `clip`.

In [ ]:
minimo, maximo = 0, 100
fuera = (df["antiguedad_anios"] < minimo) | (df["antiguedad_anios"] > maximo)
print("Fuera de rango:", fuera.sum())

df_cap = df.copy()                                    # nunca toques el original
df_cap["antiguedad_anios"] = df_cap["antiguedad_anios"].clip(lower=minimo, upper=maximo)

print("Filas antes / después:", len(df), "/", len(df_cap))           # capping NO borra filas
print("Máximo antes / después:", df["antiguedad_anios"].max(), "/", df_cap["antiguedad_anios"].max())
print("NaN antes / después:", df["antiguedad_anios"].isna().sum(), "/", df_cap["antiguedad_anios"].isna().sum())

Así lo hicieron con los **sensores de voltaje** (0-100 que llegaban a 150 en tormentas):
*"los llevaron a sus límites, para no dejar un hueco"*. Simulemos una serie de un sensor:

In [ ]:
rng = np.random.default_rng(0)
horas = pd.date_range("2024-01-01", periods=48, freq="h")
voltaje = pd.Series(rng.normal(70, 8, 48).round(1), index=horas)
voltaje.iloc[[10, 11, 30]] = [150.0, 132.0, -12.0]     # rayos durante una tormenta

eliminado = voltaje[(voltaje >= 0) & (voltaje <= 100)]  # opción 1: eliminar
capado = voltaje.clip(0, 100)                           # opción 2: llevar a límites

print("Mediciones originales:", len(voltaje))
print("Si elimino:", len(eliminado), "-> horas perdidas:", [str(h) for h in voltaje.index.difference(eliminado.index)])
print("Si hago capping:", len(capado), "-> ninguna hora perdida")

In [ ]:
# (opcional) gráfico del sensor
if HAY_PLT:
    ax = voltaje.plot(label="original", figsize=(9, 3))
    capado.plot(ax=ax, label="capping 0-100", linestyle="--")
    ax.axhline(100, color="red", linewidth=0.8)
    ax.axhline(0, color="red", linewidth=0.8)
    ax.legend()
    plt.show()

## Paso 4 · Por qué el outlier daña al modelo: la media y la recta
La **media** es el ejemplo más simple de "jalón": un solo valor extremo la mueve. La **mediana** casi no se entera.

In [ ]:
for col in ["precio_usd", "area_m2", "antiguedad_anios"]:
    media, mediana = df[col].mean(), df[col].median()
    print(f"{col:18s} media={media:12.2f}  mediana={mediana:12.2f}  diferencia={media - mediana:10.2f}")

Ahora la **recta de regresión** del profesor (*"los outliers jalan demasiado el modelo"*).
`np.polyfit(x, y, 1)` ajusta `y = a·x + b` y devuelve `[a, b]`. La pendiente `a` = USD por cada m² extra.

In [ ]:
errores = df["area_m2"] > 400            # los 12 errores de área ×10
print("Errores:", errores.sum(), "de", len(df))

a_con, b_con = np.polyfit(df["area_m2"], df["precio_usd"], 1)
a_sin, b_sin = np.polyfit(df.loc[~errores, "area_m2"], df.loc[~errores, "precio_usd"], 1)
print(f"Pendiente CON outliers: {a_con:8.2f} USD/m²")
print(f"Pendiente SIN outliers: {a_sin:8.2f} USD/m²")
print(f"Una casa de 200 m² -> con: {a_con*200 + b_con:,.0f} USD | sin: {a_sin*200 + b_sin:,.0f} USD")

In [ ]:
# (opcional) las dos rectas
if HAY_PLT:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(df["area_m2"], df["precio_usd"], s=5, alpha=0.4)
    xs = np.linspace(0, df["area_m2"].max(), 100)
    ax.plot(xs, a_con * xs + b_con, color="red", label="con outliers")
    ax.plot(xs, a_sin * xs + b_sin, color="green", label="sin outliers")
    ax.set_xlabel("area_m2")
    ax.set_ylabel("precio_usd")
    ax.legend()
    plt.show()

## ✏️ Tu turno 1
1. Repite el cálculo de pendientes, pero en lugar de **eliminar** los 12 errores, **divide su área entre 10** (corrígelos).
   ¿La pendiente se parece más a la de "sin outliers"?
2. ¿Por qué corregir es mejor que eliminar aquí? (Piensa en las otras columnas de esas 12 filas.)

In [ ]:
# Tu código aquí


## Paso 5 · "No podemos eliminar todos"
Si borramos **toda fila** con algo raro en cualquier columna, ¿cuánto perdemos? Usemos límites amplios.

In [ ]:
reglas = {
    "area_m2 > 400": df["area_m2"] > 400,
    "antiguedad > 100": df["antiguedad_anios"] > 100,
    "precio > 600 000": df["precio_usd"] > 600_000,
}
borrar = pd.Series(False, index=df.index)
for nombre, mascara in reglas.items():
    print(f"{nombre:18s}: {mascara.sum()} filas")
    borrar = borrar | mascara
print("Filas que se irían:", borrar.sum())
print("Valores SANOS de distrito/tipo/estado que se pierden con ellas:", borrar.sum() * 3)

Aquí son pocas filas, pero en datasets reales con muchas columnas *"puedo comprometer a las otras variables"*.
Y además están los **outliers multidimensionales**: un valor puede ser normal en cada columna por separado,
pero la **combinación** ser rara (por ejemplo, una casa de 300 m² con 1 sola habitación).

## Paso 6 · PowerTransformer: acercar a la normal
- **Box-Cox**: solo valores > 0.
- **Yeo-Johnson** (por defecto): acepta ceros y negativos.

`skew()` mide la asimetría: 0 = simétrica (como una normal); positivo grande = cola larga a la derecha.

In [ ]:
from sklearn.preprocessing import PowerTransformer

cols = ["precio_usd", "area_m2"]
pt = PowerTransformer(method="box-cox")      # standardize=True por defecto: media 0, desviación 1
df_pt = df.copy()
df_pt[cols] = pt.fit_transform(df[cols])

pd.DataFrame({"asimetria_antes": df[cols].skew(), "asimetria_despues": df_pt[cols].skew()}).round(2)

In [ ]:
# ¿A cuántas desviaciones de la media está la mansión, antes y después?
i = df["precio_usd"].idxmax()
z_antes = (df.loc[i, "precio_usd"] - df["precio_usd"].mean()) / df["precio_usd"].std()
print(f"Antes: {z_antes:.1f} desviaciones | Después: {df_pt.loc[i, 'precio_usd']:.1f} desviaciones")

In [ ]:
# Box-Cox con ceros falla: las oficinas tienen 0 habitaciones
try:
    PowerTransformer(method="box-cox").fit_transform(df[["habitaciones"]])
except ValueError as error:
    print("ValueError:", error)

# Yeo-Johnson sí acepta el 0 (y negativos)
hab_yj = PowerTransformer(method="yeo-johnson").fit_transform(df[["habitaciones"]])
print("Yeo-Johnson OK, media:", round(float(hab_yj.mean()), 6))

In [ ]:
# (opcional) histogramas antes / después
if HAY_PLT:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    df["precio_usd"].plot.hist(bins=60, ax=axes[0], title="precio_usd original")
    df_pt["precio_usd"].plot.hist(bins=60, ax=axes[1], title="precio_usd Box-Cox")
    plt.show()

## ✏️ Tu turno 2
1. Aplica Yeo-Johnson a `antiguedad_anios` (tiene NaN y ceros). ¿Qué pasa con los NaN? ¿Cuánto cambia `skew()`?
2. ¿Por qué no podrías usar Box-Cox en esa columna?

In [ ]:
# Tu código aquí


## Paso 7 · Correlación entre categóricas: V de Cramér
*"Correlación de categóricas entre ellas y de numéricas entre ellas: no lo mezclen."*
La V de Cramér parte de una **tabla de contingencia** (`pd.crosstab`): cuántas filas hay de cada combinación.

In [ ]:
sin_habitaciones = df["habitaciones"] == 0
pd.crosstab(df["tipo_inmueble"], sin_habitaciones)

Solo las **oficinas** tienen 0 habitaciones: saber el tipo te dice con certeza si tiene habitaciones → V = 1.
Comparemos con `tipo_inmueble` vs `estado`, donde las proporciones son parecidas en cada tipo → V cerca de 0.

In [ ]:
pd.crosstab(df["tipo_inmueble"], df["estado"], normalize="index").round(2)

La fórmula exacta de la V de Cramér es el **reto** de `ejercicio.py` (función `cramers_v`).

## ✏️ Tu turno 3
1. Arma `pd.crosstab(df["distrito"], df["tipo_inmueble"], normalize="index")`. ¿Ves algún distrito con
   proporciones muy distintas al resto? ¿Esperarías una V alta o baja?
2. Pregunta de criterio: en una serie de **ventas diarias**, el 24 de diciembre sale un pico enorme **todos los años**.
   ¿Es un outlier? ¿Qué variable agregarías al modelo para que lo "entienda"? (Pista: la respuesta del profe a Arturo.)

In [ ]:
# Tu código aquí


## Resumen
| Idea de la clase | En código |
|---|---|
| Valores fuera de límites conocidos | `(s < minimo)` o `(s > maximo)` |
| Llevar a límites (capping) | `s.clip(lower=minimo, upper=maximo)` |
| El outlier "jala" | `s.mean()` vs `s.median()` · `np.polyfit(x, y, 1)` con y sin outliers |
| Acercar a la normal | `PowerTransformer(method="box-cox" / "yeo-johnson")` |
| Asociación entre categóricas | `pd.crosstab` → V de Cramér |

Ahora resuelve `ejercicio.py` y evalúa con
`python -m framework evaluar cursos/nivelacion/n1-python-data-science/m2-outliers-encoding/clase-06-outliers`.